# POISE — PPO training (off-device / Kaggle)

🔒 Train OFF-DEVICE only. On-board RL is far too slow (thermal time-constants are minutes).
The policy trains against the calibrated RC simulator with domain randomization, then is
validated separately on the real board (step 9).

🔒 Reward-hacking guard: we sweep reward weights and report the **converged depth
distribution** for each. A policy that collapses to minimum depth is a FAILED reward,
not a success.

In [ ]:
# Install POISE + RL extras (Kaggle).
# !pip install -q -e .[rl]   # from the repo root, or: pip install stable-baselines3 gymnasium
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root if running from scripts/
from poise.config import load_config
from poise.storage.db import init_db
cfg = load_config()
print('sim params source:', cfg.sim.params_source, '(placeholder => calibrate first for a trustworthy policy)')

## Quality table
The reward's quality term must be the **measured** depth→KL table from the step-3 gate.
If the DB has no `quality_profile` rows, training falls back to a clearly-labeled
synthetic table (plumbing only) with a loud warning.

In [ ]:
from poise.rl.train_ppo import resolve_quality_table
conn = init_db(cfg.storage.db_path)
quality_table = resolve_quality_table(cfg, conn)
print('synthetic quality table?', quality_table.synthetic)

## Train one policy

In [ ]:
from poise.rl.train_ppo import train
result = train(cfg, out_path='ppo_policy.zip', conn=conn, quality_table=quality_table,
               total_timesteps=200_000, seed=0)
print('mean reward:', result.final_mean_reward)
print('converged depth distribution:', result.converged_depth_hist)
print('COLLAPSED TO MIN (failed reward)?', result.collapsed_to_min)

## Reward-weight sweep (the audit)
Report the converged depth distribution per weight config. Pick a config that holds a
healthy spread of depths AND meets the thermal target — never one that collapses to min.

In [ ]:
from poise.rl.train_ppo import sweep_weights
grid = [
    {'w_q': 0.25}, {'w_q': 1.0}, {'w_q': 2.0}, {'w_q': 4.0},
]
results = sweep_weights(cfg, grid, conn=conn, total_timesteps=100_000)
for w, r in zip(grid, results):
    print(w, '->', r.converged_depth_hist, 'collapsed:', r.collapsed_to_min)

Copy the chosen `ppo_policy.zip` to the Jetson and set `POISE_POLICY_PATH` + `POISE_CONTROL_MODE=ppo`.
Then validate on the real board (step 9) and quantify the sim-to-real gap.